# Experiment 3: scratch-trained anti-stagnation CNN-DQN

This is a separate training stage with randomly initialized policy and target networks. It does not load or overwrite Experiment 1 or Experiment 2. The reward preserves novelty and known-space BFS frontier shaping while adding stationary, immediate left-right reversal, and blocked-forward penalties.


## Imports and configuration


In [1]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch

from minigrid.core.constants import OBJECT_TO_IDX

from utils import alert_training_complete
from utils.minigrid_dual_scale_dqn import DualScaleMapDQN, DualScaleState
from utils.minigrid_experiment3 import (
    DEFAULT_COVERAGE_THRESHOLDS,
    DeadlockConfig,
    DeadlockDetector,
    Experiment3RewardConfig,
    calculate_action_source_metrics,
    compute_exploration_reward,
    evaluate_exploration_controller,
    load_experiment3_policy,
    reachable_frontier_distance,
    run_experiment_3,
    select_epsilon_greedy_action_with_source,
    summarize_evaluations,
    episode_end_reason,
    transition_is_done,
)
from utils.minigrid_mapper import MapCell


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tensor_config = {
    "local_size": 15,
    "global_size": 15,
    "unknown_padding": 2,
    "hysteresis_bins": 1,
}
conv_channels = (32, 64, 64)
fusion_hidden_size = 256

training_environment_config = {
    "width": 19,
    "height": 19,
    "min_rooms": 4,
    "max_rooms": 6,
    "min_room_size": 5,
    "doorway_width": 1,
    "max_steps": 200,
}
training_seed_start = 0
training_seed_count = 10_000
validation_seeds = list(range(20_000, 20_100))
diagnostic_validation_seeds = [validation_seeds[0], validation_seeds[50], validation_seeds[-1]]

RUN_EXPERIMENT_3 = True
RUN_GREEDY_EVALUATION = True


Device: cuda


## Focused implementation checks


In [3]:
reward_config = Experiment3RewardConfig(
    novelty_beta=1.0,
    frontier_progress_beta=0.15,
    stationary_penalty=-0.5,
    oscillation_penalty=-2.0,
    blocked_penalty=-2.0,
)
deadlock_config = DeadlockConfig(
    oscillation_deadlock_threshold=8,
    stationary_deadlock_threshold=8,
    deadlock_penalty=-1.0,
    no_discovery_deadlock_multiplier=1.0,
)


def reward_case(new_cells=0, before=3, after=3, action=0, previous_action=None,
                position_before=(0, 0), position_after=(0, 0), blocked=False):
    return compute_exploration_reward(
        mapping_update=SimpleNamespace(new_cells=new_cells),
        frontier_distance_before=before,
        frontier_distance_after=after,
        action=action,
        position_before=position_before,
        position_after=position_after,
        previous_action=previous_action,
        forward_was_known_blocked=blocked,
        config=reward_config,
    )


assert reward_case(action=0)["stationary"] == reward_config.stationary_penalty
assert reward_case(action=2, position_after=(1, 0))["stationary"] == 0.0
assert reward_case(action=1, previous_action=0)["oscillation"] == reward_config.oscillation_penalty
assert reward_case(action=0, previous_action=1)["oscillation"] == reward_config.oscillation_penalty
blocked = reward_case(action=2, blocked=True)
assert blocked["stationary"] + blocked["blocked"] == (
    reward_config.stationary_penalty + reward_config.blocked_penalty
)
discovery = reward_case(new_cells=5, before=0, after=4, action=0)
assert discovery["novelty"] == 5.0 and discovery["frontier"] == 0.0


def run_detector(actions, *, progress_step=None, discovery_step=None, movement_step=None):
    detector = DeadlockDetector(deadlock_config)
    previous_action = None
    statuses = []
    for step, action in enumerate(actions, start=1):
        status = detector.update(
            action=action,
            previous_action=previous_action,
            position_changed=step == movement_step,
            new_cells=1 if step == discovery_step else 0,
            frontier_distance_before=3,
            frontier_distance_after=2 if step == progress_step else 3,
        )
        statuses.append(status)
        previous_action = action
        if status["deadlock_triggered"]:
            break
    return statuses


alternating_eight = run_detector([0, 1, 0, 1, 0, 1, 0, 1])
assert alternating_eight[-1]["deadlock_type"] == "oscillation_deadlock"
assert not run_detector([0, 1, 0, 1])[-1]["deadlock_triggered"]
assert not run_detector([0, 1, 2], movement_step=3)[-1]["deadlock_triggered"]
assert run_detector([0, 1, 0, 2], movement_step=4)[-1]["oscillation_counter"] == 0
assert run_detector([0, 1, 0], discovery_step=3)[-1]["stationary_counter"] == 0
assert run_detector([0, 1, 0], progress_step=3)[-1]["oscillation_counter"] == 0
assert run_detector([0] * 20)[-1]["deadlock_type"] == "stationary_deadlock"
assert run_detector([2] * 20)[-1]["deadlock_type"] == "stationary_deadlock"
assert transition_is_done(False, False, "stationary_deadlock")
assert episode_end_reason(False, False, "oscillation_deadlock") == "oscillation_deadlock"
expected_no_discovery_threshold = int(
    np.ceil(deadlock_config.no_discovery_deadlock_multiplier * (19 + 19))
)
assert (
    deadlock_config.no_discovery_threshold(19, 19)
    == expected_no_discovery_threshold
)

no_discovery_detector = DeadlockDetector(
    deadlock_config, no_discovery_deadlock_threshold=4
)
no_discovery_statuses = []
for _ in range(4):
    no_discovery_statuses.append(
        no_discovery_detector.update(
            action=2,
            previous_action=2,
            position_changed=True,
            new_cells=0,
            frontier_distance_before=3,
            frontier_distance_after=3,
        )
    )
assert no_discovery_statuses[-1]["deadlock_type"] == "no_discovery_deadlock"

reset_detector = DeadlockDetector(deadlock_config, no_discovery_deadlock_threshold=4)
for step in range(4):
    reset_status = reset_detector.update(
        action=2,
        previous_action=2,
        position_changed=True,
        new_cells=1 if step == 2 else 0,
        frontier_distance_before=3,
        frontier_distance_after=3,
    )
assert not reset_status["deadlock_triggered"]
assert reset_status["steps_since_discovery"] == 1

# The shared DQN target multiplies future Q by (1 - done), so deadlock done=True
# produces the unbootstrapped target y = reward.
terminal_reward = deadlock_config.deadlock_penalty
terminal_target = terminal_reward + 0.99 * 123.0 * (1.0 - float(True))
assert terminal_target == terminal_reward
assert sum(status["deadlock_triggered"] for status in alternating_eight) == 1


class FixedBranchRng:
    def __init__(self, random_value, action_index=0):
        self.random_value = random_value
        self.action_index = action_index

    def random(self):
        return self.random_value

    def integers(self, _):
        return self.action_index


test_policy = DualScaleMapDQN().to(device)
test_state = DualScaleState(
    local=np.zeros((4, 15, 15), dtype=np.float32),
    global_map=np.zeros((5, 15, 15), dtype=np.float32),
    global_scale_log2=0.0,
)
_, random_source = select_epsilon_greedy_action_with_source(
    test_policy, test_state, 0.10, device, random_generator=FixedBranchRng(0.0, 2)
)
_, greedy_source = select_epsilon_greedy_action_with_source(
    test_policy, test_state, 0.10, device, random_generator=FixedBranchRng(1.0)
)
assert random_source == "random" and greedy_source == "greedy"

source_metrics = calculate_action_source_metrics({
    "random_action_count": 2,
    "greedy_action_count": 8,
    "random_discovery_actions": 1,
    "greedy_discovery_actions": 2,
    "random_new_cells": 4,
    "greedy_new_cells": 6,
    "random_position_changes": 1,
    "greedy_position_changes": 4,
})
assert source_metrics["random_discovery_probability"] == 0.5
assert source_metrics["greedy_discovery_probability"] == 0.25
assert source_metrics["fraction_of_new_cells_from_random"] == 0.4


class SyntheticMapper:
    def __init__(self, cells, frontiers):
        self.position = np.array([0, 0], dtype=int)
        self.cells = cells
        self._frontiers = frontiers

    def frontier_cells(self):
        return self._frontiers


empty = MapCell(OBJECT_TO_IDX["empty"], 0, 0)
wall = MapCell(OBJECT_TO_IDX["wall"], 0, 0)
blocked_mapper = SyntheticMapper(
    {(0, 0): empty, (1, 0): wall, (2, 0): empty}, {(2, 0)}
)
assert reachable_frontier_distance(blocked_mapper) is None
print("Reward, deadlock, terminal-target, source-metric, and BFS checks passed.")


Reward, deadlock, terminal-target, source-metric, and BFS checks passed.


## Scratch training


In [4]:
num_training_episodes = 2_000
gamma = 0.99
learning_rate = 1e-4
batch_size = 64
replay_capacity = 100_000
minimum_replay_size = 5_000
target_update_frequency = 1_000
gradient_clip = 10.0
epsilon_start = 0.10
epsilon_end = 0.05
epsilon_decay_steps = 150_000
validation_frequency = 100
validation_subset = validation_seeds[:20]
training_random_seed = 44

experiment_3_output_dir = Path("experiment_3")
experiment_3_model_path = experiment_3_output_dir / "final_model.pth"
print("Experiment 3 output:", experiment_3_output_dir.resolve())


Experiment 3 output: C:\Users\Lenovo\Documents\GitHub\Mobile-Robotics\experiment_3


In [5]:
if RUN_EXPERIMENT_3:
    policy_net, experiment_3_checkpoint = run_experiment_3(
        output_dir=experiment_3_output_dir,
        environment_config=training_environment_config,
        tensor_config=tensor_config,
        validation_seeds=validation_subset,
        diagnostic_validation_seeds=diagnostic_validation_seeds,
        device=device,
        reward_config=reward_config,
        deadlock_config=deadlock_config,
        num_training_episodes=num_training_episodes,
        training_seed_start=training_seed_start,
        training_seed_count=training_seed_count,
        training_random_seed=training_random_seed,
        conv_channels=conv_channels,
        fusion_hidden_size=fusion_hidden_size,
        learning_rate=learning_rate,
        gamma=gamma,
        batch_size=batch_size,
        replay_capacity=replay_capacity,
        minimum_replay_size=minimum_replay_size,
        target_update_frequency=target_update_frequency,
        gradient_clip=gradient_clip,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        epsilon_decay_steps=epsilon_decay_steps,
        validation_frequency=validation_frequency,
    )
    alert_training_complete("Experiment 3 scratch training finished.")
else:
    print("Ready. Set RUN_EXPERIMENT_3 = True to start the 2,000-episode run.")


Episode   25/2000 | coverage=14.9% | reward=-3.77 | novelty=19.92 | frontier=-0.09 | epsilon=0.100 | loss=nan | move=26.4% | osc=0.0% | deadlock=100.0% | Pdisc(greedy)=12.4% | Pdisc(random)=29.3%
Episode   50/2000 | coverage=19.5% | reward=5.10 | novelty=31.40 | frontier=-0.12 | epsilon=0.100 | loss=nan | move=32.4% | osc=0.1% | deadlock=100.0% | Pdisc(greedy)=13.9% | Pdisc(random)=32.3%
Episode   75/2000 | coverage=17.7% | reward=0.75 | novelty=25.16 | frontier=-0.13 | epsilon=0.100 | loss=nan | move=32.9% | osc=0.0% | deadlock=100.0% | Pdisc(greedy)=12.7% | Pdisc(random)=28.2%
Episode  100/2000 | coverage=17.4% | reward=1.73 | novelty=25.40 | frontier=-0.17 | epsilon=0.099 | loss=nan | move=33.8% | osc=0.0% | deadlock=100.0% | Pdisc(greedy)=13.1% | Pdisc(random)=35.3%
Validation 100: coverage=11.7% | median=12.0% | zero-info=91.1% | movement=27.0% | stationary=73.0% | oscillation=0.0% | osc-deadlock=0.0% | stationary-deadlock=100.0% | no-discovery=0.0% | time-limit=0.0% | 50%=0.0% | 

## Greedy validation


In [6]:
if RUN_GREEDY_EVALUATION:
    if not experiment_3_model_path.exists():
        raise FileNotFoundError(experiment_3_model_path.resolve())
    evaluation_policy, _ = load_experiment3_policy(experiment_3_model_path, device)
    greedy_results = evaluate_exploration_controller(
        validation_seeds,
        training_environment_config,
        tensor_config,
        device,
        reward_config,
        deadlock_config=deadlock_config,
        policy_net=evaluation_policy,
    )
    greedy_summary = summarize_evaluations(greedy_results)
    assert greedy_summary["random_action_count"] == 0
    print(greedy_summary)
else:
    print("Greedy evaluation remains disabled until an Experiment 3 model exists.")


{'mean_final_coverage': 0.4966759002770084, 'median_final_coverage': 0.4736842105263158, 'mean_traversable_coverage': 0.5149129640620839, 'new_cells_per_step': 3.0004488826090245, 'zero_information_ratio': 0.6568496177613683, 'stationary_action_ratio': 0.4555593804592494, 'oscillation_ratio': 0.19626122246231556, 'blocked_forward_ratio': 0.01354911014911015, 'position_change_ratio': 0.5444406195407506, 'mean_maximum_consecutive_stationary_steps': 6.01, 'random_action_count': 0, 'oscillation_deadlock_count': 43, 'stationary_deadlock_count': 16, 'no_discovery_deadlock_count': 41, 'time_limit_count': 0, 'environment_terminated_count': 0, 'fraction_oscillation_deadlock': 0.43, 'fraction_stationary_deadlock': 0.16, 'fraction_no_discovery_deadlock': 0.41, 'fraction_time_limit': 0.0, 'fraction_reaching_50': 0.49, 'mean_steps_to_50_if_reached': 29.591836734693878, 'fraction_reaching_75': 0.16, 'mean_steps_to_75_if_reached': 54.4375, 'fraction_reaching_90': 0.06, 'mean_steps_to_90_if_reached': 